In [1]:
import pandas as pd
import os
import subprocess
import shutil
import random
from pathlib import Path
from dotenv import load_dotenv, set_key
import requests
import stat
import time
import re

# === CONFIGURATION ===
MAX_PROJECTS = 3000
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150
ENV_FILE = 'All_tokens.env'
SEPARATOR = "__"

# === LOAD .env ===
load_dotenv(ENV_FILE)
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
headers = {'Authorization': f'token {GITHUB_TOKEN}'}

if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in All_tokens.env")

START_NUMBER = int(os.getenv("START_NUMBER"))
SAMPLE_LIST_RAW = os.getenv("SAMPLE_LIST", "").strip()

# === PATHS ===
csv_path = Path(r"C:\Android Mobile App\Android_Repos_MultiRange.csv")
base_dir = Path(r"C:\Android Mobile App\6.2-Shallow_Clone")
clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
yml_output_dir = base_dir / "Config Files"
commits_dir = base_dir / "Commits"
build_info_dir = base_dir / "BuildInfo"
metadata_path = base_dir / "8.2-Project_Metadata.csv"
config_location_csv = base_dir / "Config_Location.csv"
log_path = base_dir / "repo_processing.log"
failed_projects = []

# === ENSURE ALL FOLDERS EXIST ===
for path in [clone_dir, yml_output_dir, commits_dir, build_info_dir, cloned_sample_dir]:
    path.mkdir(parents=True, exist_ok=True)

# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['clone_url'].notna()]
df['github_url'] = df['clone_url'].astype(str).str.strip()
df = df[df['clone_url'].str.startswith("https://")]
df[['clone_url']].to_csv(base_dir / 'Sorted_URL_List.csv', index_label='Index')

# === HANDLE SAMPLE_LIST ===
if SAMPLE_LIST_RAW:
    sample_indices_to_keep = set(map(int, SAMPLE_LIST_RAW.split(',')))
    print(f"🔁 Loaded SAMPLE_LIST from .env with {len(sample_indices_to_keep)} indices.")
else:
    random.seed(RANDOM_SEED)
    sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))
    sample_string = ",".join(map(str, sorted(sample_indices_to_keep)))
    set_key(ENV_FILE, 'SAMPLE_LIST', sample_string)
    print(f"🎲 Generated and saved new SAMPLE_LIST with {len(sample_indices_to_keep)} indices.")

# === LOAD EXISTING CONFIG LOCATIONS IF RESUMING ===
if config_location_csv.exists():
    config_locations_df = pd.read_csv(config_location_csv)
else:
    config_locations_df = pd.DataFrame(columns=["repo_name", "config_file_path", "file_type"])

# === Logger ===
def log_message(message):
    print(message)
    with open(log_path, "a", encoding="utf-8") as log_file:
        log_file.write(message + "\n")

def check_rate_limit():
    response = requests.get("https://api.github.com/rate_limit", headers=headers)
    if response.status_code == 200:
        remaining = response.json()['rate']['remaining']
        reset_time = response.json()['rate']['reset']
        if remaining < 50:
            wait_seconds = reset_time - int(time.time())
            if wait_seconds > 0:
                log_message(f"⏳ Rate limit low. Sleeping for {wait_seconds} seconds...")
                time.sleep(wait_seconds + 5)
        return remaining
    else:
        log_message("⚠️ Could not check rate limit.")
        return None

# === Strict CI patterns ===
CI_CONFIG_PATTERNS = [
    re.compile(r'\.github/workflows/.*\.(yml|yaml)$'),
    re.compile(r'\.travis\.yml$'),
    re.compile(r'(\.circleci/config\.yml|circle\.yml)$'),
    re.compile(r'\.gitlab-ci\.yml$'),
    re.compile(r'azure-pipelines\.yml$'),
    re.compile(r'(\.appveyor\.yml|appveyor\.yml)$'),
    re.compile(r'bitbucket-pipelines\.yml$'),
]

# === Extract commits ===
def extract_commit_metadata_api(owner, repo, repo_name, max_commits=500):
    url = f"https://api.github.com/repos/{owner}/{repo}/commits"
    commits = []
    page = 1
    check_rate_limit()

    while len(commits) < max_commits:
        paged_url = f"{url}?per_page=100&page={page}"
        response = requests.get(paged_url, headers=headers)
        if response.status_code != 200:
            log_message(f"❌ Error fetching {owner}/{repo} commits: {response.status_code}")
            break

        data = response.json()
        if not data:
            break

        for commit in data:
            commit_data = commit.get("commit", {})
            author = commit_data.get("author", {})
            commits.append({
                "sha": commit.get("sha"),
                "author_name": author.get("name"),
                "author_email": author.get("email"),
                "date": author.get("date"),
                "message": commit_data.get("message"),
                "html_url": commit.get("html_url")
            })

            if len(commits) >= max_commits:
                break

        page += 1

    if commits:
        df_commits = pd.DataFrame(commits)
        filename = f"{repo_name}{SEPARATOR}commits.csv"
        df_commits.to_csv(commits_dir / filename, index=False)
        print(f"✅ API commit data saved for {owner}/{repo}")
    else:
        log_message(f"⚠️ No commits found for {owner}/{repo} via API")
        failed_projects.append(repo_name)

# === Extract contributors ===
def extract_contributor_logins(owner, repo, repo_name):
    url = f"https://api.github.com/repos/{owner}/{repo}/contributors"
    logins = []
    page = 1
    check_rate_limit()

    while True:
        paged_url = f"{url}?per_page=100&page={page}"
        response = requests.get(paged_url, headers=headers)
        if response.status_code != 200:
            log_message(f"❌ Error fetching {owner}/{repo} contributors: {response.status_code}")
            break

        data = response.json()
        if not data:
            break

        logins.extend([c.get("login") for c in data if c.get("login")])

        if len(data) < 100:
            break

        page += 1

    if logins:
        contributors_text = "\n".join(logins)
        filename = f"{repo_name}{SEPARATOR}contributors.txt"
        contributors_path = commits_dir / filename
        with open(contributors_path, "w", encoding="utf-8") as f:
            f.write(contributors_text)
        print(f"👥 Saved contributors to: {contributors_path.name}")
    else:
        log_message(f"⚠️ No contributors found for {owner}/{repo}")

# === PROCESS EACH REPO ===
for i in range(START_NUMBER - 1, len(df)):
    url = df.iloc[i]['github_url']
    parts = url.split('/')
    if len(parts) < 5:
        continue
    username, project = parts[-2], parts[-1].replace('.git', '')

    repo_index = str(i).zfill(4)
    repo_name = f"{repo_index}{SEPARATOR}{username}{SEPARATOR}{project}"

    repo_path = clone_dir / repo_name

    print(f"\n🔍 [{i+1}/{len(df)}] Processing {repo_name}...")

    try:
        subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', url, str(repo_path)],
                       check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print("✅ Clone complete")
    except Exception as e:
        log_message(f"❌ Clone failed for {repo_name}: {e}")
        failed_projects.append(repo_name)
        continue

    # === Checkout default branch ===
    try:
        base_api = f"https://api.github.com/repos/{username}/{project}"
        check_rate_limit()
        r_branch = requests.get(base_api, headers=headers, timeout=15)
        if r_branch.status_code == 200:
            default_branch = r_branch.json().get('default_branch', 'main')
            subprocess.run(["git", "-C", str(repo_path), "checkout", default_branch],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print(f"📌 Checked out default branch: {default_branch}")
        else:
            print(f"⚠️ Could not detect default branch for {repo_name}, using current HEAD")
    except Exception as e:
        log_message(f"⚠️ Failed to checkout default branch for {repo_name}: {e}")

    # === Extract commits ===
    extract_commit_metadata_api(username, project, repo_name)

    # === Extract contributors ===
    extract_contributor_logins(username, project, repo_name)

    # === Scan config/build files (STRICT: workflow folders only) ===
    ci_keywords = ['ci', 'build', 'test', 'workflow', 'pipeline', 'instrumentation']
    config_files_found = []

    for root, _, files in os.walk(repo_path):
        for file in files:
            file_lower = file.lower()
            file_path = Path(root) / file
            rel_path = str(file_path.relative_to(repo_path)).replace("\\", "/")

            try:
                should_copy = False
                file_type = file_lower.split('.')[-1]

                if file_lower.endswith(('.yml', '.yaml')) and any(pattern.search(rel_path) for pattern in CI_CONFIG_PATTERNS):
                    should_copy = True

                elif file_lower.endswith(('build.gradle', 'build.gradle.kts')):
                    should_copy = True

                elif file_lower.endswith(('.json', '.sh')):
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read().lower()
                        if any(keyword in content for keyword in ci_keywords):
                            should_copy = True

                if should_copy:
                    flat_filename = f"{repo_name}{SEPARATOR}{file}"
                    if file_lower.endswith('build.gradle'):
                        destination_path = build_info_dir / flat_filename
                    else:
                        destination_path = yml_output_dir / flat_filename
                    shutil.copy2(file_path, destination_path)
                    config_files_found.append({
                        "repo_name": repo_name,
                        "config_file_path": flat_filename,
                        "file_type": file_type
                    })

            except Exception as e:
                log_message(f"⚠️ Could not process or copy {rel_path} in {repo_name}: {e}")

    if config_files_found:
        config_locations_df = pd.concat([config_locations_df, pd.DataFrame(config_files_found)], ignore_index=True)

    # === Move or delete ===
    try:
        if i in sample_indices_to_keep:
            dest_path = cloned_sample_dir / repo_path.name
            if dest_path.exists():
                shutil.rmtree(dest_path, onerror=lambda f, p, _: os.chmod(p, stat.S_IWRITE))
            shutil.move(str(repo_path), str(dest_path))
            print(f"📆 Sample repo moved to: {dest_path}")
        else:
            shutil.rmtree(repo_path, onerror=lambda f, p, _: os.chmod(p, stat.S_IWRITE))
            print(f"🗑️ Repo deleted: {repo_name}")
    except Exception as e:
        log_message(f"❌ Error moving/deleting repo folder for {repo_name}: {e}")

    set_key(ENV_FILE, 'START_NUMBER', str(i + 2))
    config_locations_df.drop_duplicates().to_csv(config_location_csv, index=False)

if failed_projects:
    failed_df = pd.DataFrame(failed_projects, columns=["repo_name"])
    failed_df.to_csv(base_dir / "Failed_Projects.csv", index=False)
    print(f"\n⚠️ {len(failed_projects)} projects failed. Saved to Failed_Projects.csv.")

print(f"\n✅ Process complete. Sampled: {len(sample_indices_to_keep)} | Total Processed: {len(df) - (START_NUMBER - 1)}")
print("\n✅ All selected repositories have been processed.")


🔁 Loaded SAMPLE_LIST from .env with 150 indices.

🔍 [1/2614] Processing 0000__JunkFood02__Seal...
✅ Clone complete
📌 Checked out default branch: main
✅ API commit data saved for JunkFood02/Seal
👥 Saved contributors to: 0000__JunkFood02__Seal__contributors.txt
🗑️ Repo deleted: 0000__JunkFood02__Seal

🔍 [2/2614] Processing 0001__android__nowinandroid...
✅ Clone complete
📌 Checked out default branch: main
✅ API commit data saved for android/nowinandroid
👥 Saved contributors to: 0001__android__nowinandroid__contributors.txt
🗑️ Repo deleted: 0001__android__nowinandroid

🔍 [3/2614] Processing 0002__square__picasso...
✅ Clone complete
📌 Checked out default branch: master
✅ API commit data saved for square/picasso
👥 Saved contributors to: 0002__square__picasso__contributors.txt
🗑️ Repo deleted: 0002__square__picasso

🔍 [4/2614] Processing 0003__google__flexbox-layout...
✅ Clone complete
📌 Checked out default branch: main
✅ API commit data saved for google/flexbox-layout
👥 Saved contributors to